In [ ]:
import os
os.makedirs("/content/experiments", exist_ok=True)
!pip install trimesh scikit-image plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 10.3 MB/s eta 0:00:00


In [ ]:
%%writefile /content/experiments/cvr_train.cpp
#pragma GCC optimize("O3,unroll-loops")
#pragma GCC target("avx2,bmi,bmi2,lzcnt,popcnt")

#include <iostream>
#include <vector>
#include <cmath>
#include <numeric>
#include <algorithm>
#include <random>
#include <unordered_set>
#include <fstream>
#include <sstream>
#include <string>
#include <iomanip>
using namespace std;

class CoreVectorRegression {
public:
    double C, mu, gamma, epsilon;
    int max_iter;
    double b, epsilon_bar;
    vector<int>    core_idx;
    vector<double> alpha;
    vector<vector<double>> core_cache;
    vector<double> diag_all, delta_all;

private:
    // SỬA LỖI: Dùng Laplacian Kernel thay vì RBF (Đúng như mô tả trong bài báo 4.2)
    // k(x,y) = exp(-gamma * ||x-y||) chứ không phải bình phương ||x-y||^2
    inline double laplacian(const vector<double>& x1, const vector<double>& x2) const {
        double dist = 0;
        const double* p1 = x1.data();
        const double* p2 = x2.data();
        for (int i = 0, n = (int)x1.size(); i < n; ++i) {
            double d = p1[i]-p2[i]; dist += d*d;
        }
        return exp(-gamma * sqrt(dist));
    }

    void solve_smo(int k, double reg_diag, const vector<double>& d_S,
                   vector<double>& g, double tol,
                   const vector<vector<double>>& K_core) {
        int n = 2 * k;
        for (int i = 0; i < n; ++i) g[i] = -0.5 * d_S[i];
        for (int j = 0; j < n; ++j) {
            if (alpha[j] < 1e-12) continue;
            int cj = j >> 1, sj = j & 1;
            const double* cj_row = K_core[cj].data();
            double aj = alpha[j];
            for (int ct = 0; ct < k; ++ct) {
                double kv   = cj_row[ct] + 1.0;
                double diag = (ct == cj) ? reg_diag : 0.0;
                if (sj == 0) {
                    g[2*ct]   += (kv + diag) * aj;
                    g[2*ct+1] += (-kv)        * aj;
                } else {
                    g[2*ct]   += (-kv)        * aj;
                    g[2*ct+1] += (kv + diag)  * aj;
                }
            }
        }

        for (int iter = 0; iter < 2000000; ++iter) {
            int ii = -1, ji_min = -1;
            double max_g = -1e18, min_g = 1e18;

            for (int t = 0; t < n; ++t) {
                if (alpha[t] > 1e-12 && g[t] > max_g) { max_g = g[t]; ii = t; }
                if (g[t] < min_g)                      { min_g = g[t]; ji_min = t; }
            }
            if (ii < 0 || ji_min < 0 || max_g - min_g < tol) break;

            int ji = -1;
            double max_obj = -1e18;
            int ci = ii>>1;
            const double* ci_row = K_core[ci].data();
            double kii_ws = ci_row[ci] + 1.0 + reg_diag;
            for (int t = 0; t < n; ++t) {
                if (max_g - g[t] > 0) {
                    int ct = t>>1;
                    double ktt = K_core[ct][ct] + 1.0 + reg_diag;
                    double kit_base = ci_row[ct] + 1.0;
                    double kit = kit_base;
                    if ((ii&1) != (t&1)) kit = -kit_base;
                    else if (ci == ct) kit += reg_diag;

                    double quad_coef = kii_ws + ktt - 2.0 * kit;
                    if (quad_coef <= 1e-12) quad_coef = 1e-12;

                    double obj = (max_g - g[t]) * (max_g - g[t]) / quad_coef;
                    if (obj > max_obj) {
                        max_obj = obj;
                        ji = t;
                    }
                }
            }
            if (ji < 0) ji = ji_min;

            int cj = ji>>1, sj = ji&1;
            int si = ii&1;
            double kii = K_core[ci][ci] + 1.0 + reg_diag;
            double kjj = K_core[cj][cj] + 1.0 + reg_diag;
            double kij_base = ci_row[cj] + 1.0;
            double kij;
            if (si == sj) {
                kij = kij_base + (ci == cj ? reg_diag : 0.0);
            } else {
                kij = -kij_base;
            }

            double eta = kii + kjj - 2.0 * kij;
            if (eta < 1e-12) eta = 1e-12;
            double lam = min((max_g - g[ji]) / eta, alpha[ii]);
            if (lam < 1e-12) break;

            alpha[ii] -= lam;
            alpha[ji] += lam;

            const double* cj_row2 = K_core[cj].data();
            for (int ct = 0; ct < k; ++ct) {
                double kvi  = ci_row[ct] + 1.0;
                double kvj  = cj_row2[ct] + 1.0;
                double di   = (ct == ci) ? reg_diag : 0.0;
                double dj   = (ct == cj) ? reg_diag : 0.0;

                double Ki_pos = (si==0) ? (kvi+di) : (-kvi);
                double Ki_neg = (si==0) ? (-kvi)   : (kvi+di);
                double Kj_pos = (sj==0) ? (kvj+dj) : (-kvj);
                double Kj_neg = (sj==0) ? (-kvj)   : (kvj+dj);

                g[2*ct]   += lam * (Kj_pos - Ki_pos);
                g[2*ct+1] += lam * (Kj_neg - Ki_neg);
            }
        }

        double sum = 0.0;
        for (double& a : alpha) { a = max(0.0, a); sum += a; }
        if (sum > 0) for (double& a : alpha) a /= sum;
    }

public:
    CoreVectorRegression(double C_, double mu_, double gamma_,
                         double eps_, int max_iter_)
        : C(C_), mu(mu_), gamma(gamma_), epsilon(eps_),
          max_iter(max_iter_), b(0), epsilon_bar(0) {}

    void fit(const vector<vector<double>>& X, const vector<double>& y) {
        int m = (int)X.size();
        diag_all.assign(2*m, 0.0);
        delta_all.assign(2*m, 0.0);
        vector<double> lin_all(2*m);
        double reg_diag = mu * m / C;
        double min_lin = 1e18, max_diag = -1e18;

        for (int i = 0; i < m; ++i) {
            double dv = 2.0 + reg_diag;
            diag_all[i] = diag_all[i+m] = dv;
            lin_all[i]   =  (2.0/C) * y[i];
            lin_all[i+m] = -(2.0/C) * y[i];
            min_lin  = min({min_lin,  lin_all[i], lin_all[i+m]});
            max_diag = max(max_diag, dv);
        }

        double eta_global = max_diag - min_lin + 1.0;
        for (int i = 0; i < 2*m; ++i)
            delta_all[i] = max(-diag_all[i] + eta_global + lin_all[i], 0.0);

        mt19937 rng(42);
        vector<int> pool(m); iota(pool.begin(), pool.end(), 0);
        shuffle(pool.begin(), pool.end(), rng);
        core_idx = {pool[0], pool[1]};
        unordered_set<int> core_set(core_idx.begin(), core_idx.end());

        for (int c : core_idx) {
            vector<double> row(m);
            for (int i = 0; i < m; ++i) row[i] = laplacian(X[c], X[i]);
            core_cache.push_back(row);
        }
        alpha.assign(4, 0.25);

        vector<vector<double>> K_core(2, vector<double>(2));
        for(int i=0; i<2; ++i) {
            for(int j=0; j<2; ++j) {
                K_core[i][j] = core_cache[i][core_idx[j]];
            }
        }

        vector<double> g;
        for (int outer = 0; outer < max_iter; ++outer) {
            int k = (int)core_idx.size();
            int n = 2 * k;

            vector<double> d_S(n);
            for (int i = 0; i < k; ++i) {
                int gi      = core_idx[i];
                double sk   = core_cache[i][gi] + 1.0 + reg_diag;
                d_S[2*i]    = sk + delta_all[gi]   - eta_global;
                d_S[2*i+1]  = sk + delta_all[gi+m] - eta_global;
            }

            g.assign(n, 0.0);
            solve_smo(k, reg_diag, d_S, g, 1e-6, K_core);

            double sum_ag = 0.0, sum_adS = 0.0;
            for (int i = 0; i < n; ++i) {
                sum_ag  += alpha[i] * g[i];
                sum_adS += alpha[i] * d_S[i];
            }
            double quad_term = sum_ag + 0.5 * sum_adS;

            b = 0.0;
            for (int i = 0; i < k; ++i) b += C * (alpha[2*i] - alpha[2*i+1]);

            double linear_term = 0.0;
            for (int i = 0; i < n; ++i) {
                int ci = i>>1, si = i&1;
                linear_term += alpha[i] * (si==0 ? y[core_idx[ci]] : -y[core_idx[ci]]);
            }
            epsilon_bar = max(linear_term - C * quad_term, 0.0);
            double c_norm_sq = quad_term;

            double R2 = max(0.5*sum_adS + eta_global - sum_ag, 1e-12);
            double threshold = pow((1.0 + epsilon) * sqrt(R2), 2);

            double sum_diff = 0.0;
            for (int j = 0; j < k; ++j) sum_diff += alpha[2*j] - alpha[2*j+1];

            auto get_kt = [&](int idx) {
                double v = sum_diff;
                for (int j = 0; j < k; ++j) {
                    double ad = alpha[2*j] - alpha[2*j+1];
                    if (abs(ad) > 1e-12) v += core_cache[j][idx] * ad;
                }
                return v;
            };

            vector<int> non_core;
            non_core.reserve(m - k);
            for (int i = 0; i < m; ++i) if (!core_set.count(i)) non_core.push_back(i);

            bool added = false;
            int ss = min(59, (int)non_core.size());
            if (ss > 0) {
                shuffle(non_core.begin(), non_core.end(), rng);
                double bd = -1; int bi = -1;
                for (int s = 0; s < ss; ++s) {
                    int idx = non_core[s];
                    double v  = get_kt(idx);
                    double d0 = c_norm_sq - 2.0*v + diag_all[idx]   + delta_all[idx];
                    double d1 = c_norm_sq + 2.0*v + diag_all[idx+m] + delta_all[idx+m];
                    double d  = max(d0, d1);
                    if (d > threshold && d > bd) { bd = d; bi = idx; }
                }
                if (bi >= 0) {
                    core_idx.push_back(bi); core_set.insert(bi);
                    vector<double> row(m);
                    for (int i = 0; i < m; ++i) row[i] = laplacian(X[bi], X[i]);
                    core_cache.push_back(row);
                    alpha.push_back(0.0); alpha.push_back(0.0);

                    int new_k = core_idx.size();
                    K_core.push_back(vector<double>(new_k));
                    for(int i=0; i<new_k-1; ++i) {
                        double val = core_cache[new_k-1][core_idx[i]];
                        K_core[new_k-1][i] = val;
                        K_core[i].push_back(val);
                    }
                    K_core[new_k-1][new_k-1] = core_cache[new_k-1][bi];
                    added = true;
                }
            }
            if (added) continue;

            double bd_all = -1; int bi_all = -1;
            for (int idx = 0; idx < m; ++idx) {
                if (core_set.count(idx)) continue;
                double v  = get_kt(idx);
                double d0 = c_norm_sq - 2.0*v + diag_all[idx]   + delta_all[idx];
                double d1 = c_norm_sq + 2.0*v + diag_all[idx+m] + delta_all[idx+m];
                double d  = max(d0, d1);
                if (d > threshold && d > bd_all) { bd_all = d; bi_all = idx; }
            }

            if (bi_all < 0) break;

            core_idx.push_back(bi_all); core_set.insert(bi_all);
            vector<double> row(m);
            for (int i = 0; i < m; ++i) row[i] = laplacian(X[bi_all], X[i]);
            core_cache.push_back(row);
            alpha.push_back(0.0); alpha.push_back(0.0);

            int new_k = core_idx.size();
            K_core.push_back(vector<double>(new_k));
            for(int i=0; i<new_k-1; ++i) {
                double val = core_cache[new_k-1][core_idx[i]];
                K_core[new_k-1][i] = val;
                K_core[i].push_back(val);
            }
            K_core[new_k-1][new_k-1] = core_cache[new_k-1][bi_all];
        }
    }
};

bool load_libsvm(const string& path, int nf,
                 vector<vector<double>>& X, vector<double>& y) {
    ifstream f(path);
    if (!f.is_open()) return false;
    string line;
    while (getline(f, line)) {
        if (line.empty()) continue;
        stringstream ss(line);
        double lbl; ss >> lbl; y.push_back(lbl);
        vector<double> x(nf, 0.0); string tok;
        while (ss >> tok) {
            auto p = tok.find(':');
            if (p != string::npos) {
                int idx = stoi(tok.substr(0, p));
                if (idx >= 1 && idx <= nf) x[idx-1] = stod(tok.substr(p+1));
            }
        }
        X.push_back(x);
    }
    return true;
}

int main(int argc, char* argv[]) {
    if (argc < 8) return 1;
    string train_path = argv[1];
    int    nf      = stoi(argv[2]);
    double C       = stod(argv[3]);
    double mu      = stod(argv[4]);
    double gamma   = stod(argv[5]);
    double epsilon = stod(argv[6]);
    string out_path = argv[7];

    vector<vector<double>> X; vector<double> y;
    load_libsvm(train_path, nf, X, y);

    CoreVectorRegression model(C, mu, gamma, epsilon, 100000);
    model.fit(X, y);

    ofstream out(out_path);
    out << setprecision(10)
        << model.b << "\n"
        << model.epsilon_bar << "\n"
        << model.core_idx.size() << "\n";
    for (size_t i = 0; i < model.core_idx.size(); ++i)
        out << model.core_idx[i] << " "
            << C * (model.alpha[2*i] - model.alpha[2*i+1]) << "\n";
    out.close();
    return 0;
}


Writing /content/experiments/cvr_train.cpp


In [ ]:
!g++ -O3 -o /content/experiments/cvr_train /content/experiments/cvr_train.cpp -lm
!chmod +x /content/experiments/cvr_train

In [ ]:
import os
import urllib.request
import tarfile
import gzip
import shutil
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════════
# HÀM ĐỌC PLY
# ═══════════════════════════════════════════════════════════════════════════════
def load_ply_points(filepath):
    points = []
    with open(filepath, 'r') as f:
        lines = f.readlines()

    header_ended = False
    vertex_count = 0
    for i, line in enumerate(lines):
        line = line.strip()
        if line.startswith('element vertex'):
            vertex_count = int(line.split()[-1])
        elif line == 'end_header':
            header_ended = True
            data_start_idx = i + 1
            break

    for i in range(data_start_idx, data_start_idx + vertex_count):
        parts = lines[i].split()
        x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
        points.append([x, y, z])

    return np.array(points)

# ═══════════════════════════════════════════════════════════════════════════════
# CẤU HÌNH DATASETS
# ═══════════════════════════════════════════════════════════════════════════════
DATASETS = {
    'bunny': {
        'name': 'Stanford Bunny',
        'url': 'http://graphics.stanford.edu/pub/3Dscanrep/bunny.tar.gz',
        'archive': 'bunny.tar.gz',
        'ply_path': os.path.join('bunny', 'reconstruction', 'bun_zipper.ply'),
        'archive_type': 'tar.gz',
    },
    'armadillo': {
        'name': 'Armadillo',
        'url': 'http://graphics.stanford.edu/pub/3Dscanrep/armadillo/Armadillo.ply.gz',
        'archive': 'Armadillo.ply.gz',
        'ply_path': 'Armadillo.ply',
        'archive_type': 'gz',
    },
    'dragon': {
        'name': 'Stanford Dragon',
        'url': 'http://graphics.stanford.edu/pub/3Dscanrep/dragon/dragon_recon.tar.gz',
        'archive': 'dragon_recon.tar.gz',
        'ply_path': os.path.join('dragon_recon', 'dragon_vrip.ply'),
        'archive_type': 'tar.gz',
    },
    'happy': {
        'name': 'Happy Buddha',
        'url': 'http://graphics.stanford.edu/pub/3Dscanrep/happy/happy_recon.tar.gz',
        'archive': 'happy_recon.tar.gz',
        'ply_path': os.path.join('happy_recon', 'happy_vrip.ply'),
        'archive_type': 'tar.gz',
    },
}

# ═══════════════════════════════════════════════════════════════════════════════
# HÀM TẢI VÀ GIẢI NÉN
# ═══════════════════════════════════════════════════════════════════════════════
def download_and_load(dataset_key, data_dir="/content/experiments/data"):
    """Tải dataset, giải nén, trả về mảng điểm 3D."""
    os.makedirs(data_dir, exist_ok=True)
    cfg = DATASETS[dataset_key]

    archive_path = os.path.join(data_dir, cfg['archive'])
    ply_path = os.path.join(data_dir, cfg['ply_path'])

    # Tải nếu chưa có
    if not os.path.exists(archive_path):
        print(f"Đang tải {cfg['name']} từ {cfg['url']}...")
        urllib.request.urlretrieve(cfg['url'], archive_path)
        print("Tải xong!")

    # Giải nén nếu chưa có file PLY
    if not os.path.exists(ply_path):
        print(f"Đang giải nén {cfg['archive']}...")
        if cfg['archive_type'] == 'tar.gz':
            with tarfile.open(archive_path) as tar:
                tar.extractall(path=data_dir)
        elif cfg['archive_type'] == 'gz':
            with gzip.open(archive_path, 'rb') as f_in:
                with open(ply_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)

    pts = load_ply_points(ply_path)
    print(f"Đã nạp {cfg['name']}: {len(pts):,} điểm 3D")
    return pts

# ═══════════════════════════════════════════════════════════════════════════════
# HÀM TIỀN XỬ LÝ: SUBSAMPLE + NORMALIZE
# ═══════════════════════════════════════════════════════════════════════════════
def preprocess_points(pts, max_points=10000, seed=42):
    """Subsample + chuẩn hóa tọa độ về [-0.5, 0.5]."""
    # 1. Subsample nếu quá nhiều điểm
    if len(pts) > max_points:
        np.random.seed(seed)
        idx = np.random.choice(len(pts), max_points, replace=False)
        pts = pts[idx]
        print(f"  → Subsample: {max_points:,} điểm")
    else:
        print(f"  → Giữ nguyên: {len(pts):,} điểm")

    # 2. Chuẩn hóa về [-0.5, 0.5] (center + scale)
    pts_min = np.min(pts, axis=0)
    pts_max = np.max(pts, axis=0)
    center = (pts_max + pts_min) / 2.0
    max_extent = np.max(pts_max - pts_min)
    pts = (pts - center) / max_extent
    print(f"  → Normalize: bbox [{pts.min():.3f}, {pts.max():.3f}]")

    return pts

# ═══════════════════════════════════════════════════════════════════════════════
# CHẠY: CHỌN DATASET VÀ NẠP
# ═══════════════════════════════════════════════════════════════════════════════
DATASET = 'dragon'   # ← Đổi thành 'armadillo', 'dragon', 'happy' để thử dataset khác
MAX_POINTS = 30000   # ← Số điểm tối đa (tăng nếu đủ RAM)

pts_raw = download_and_load(DATASET)
dataset_name = DATASETS[DATASET]['name']
ply_path = os.path.join("/content/experiments/data", DATASETS[DATASET]['ply_path'])

# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 1: Mesh gốc (nếu PLY có face data)
# ═══════════════════════════════════════════════════════════════════════════════
import plotly.graph_objects as go

try:
    import trimesh
    mesh_real = trimesh.load(ply_path)
    if hasattr(mesh_real, 'faces') and len(mesh_real.faces) > 0:
        v_real = mesh_real.vertices
        f_real = mesh_real.faces
        fig_mesh = go.Figure(data=[
            go.Mesh3d(
                x=v_real[:, 0], y=v_real[:, 1], z=v_real[:, 2],
                i=f_real[:, 0], j=f_real[:, 1], k=f_real[:, 2],
                color='white', opacity=1.0,
                lighting=dict(ambient=0.4, diffuse=0.8, roughness=0.1, specular=0.5, fresnel=0.1),
                lightposition=dict(x=100, y=100, z=100)
            )
        ])
        fig_mesh.update_layout(
            title=f"{dataset_name} - Original Mesh ({len(v_real):,} vertices)",
            font=dict(color='white'),
            paper_bgcolor='black', plot_bgcolor='black',
            scene=dict(bgcolor='black',
                       xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
                       aspectmode='data'),
            margin=dict(l=0, r=0, b=0, t=40)
        )
        fig_mesh.show()
    else:
        print(f"  (PLY không có face data → bỏ qua mesh visualization)")
except ImportError:
    print("  (trimesh chưa cài → bỏ qua mesh visualization)")

# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 2: Point Cloud gốc (trước preprocess, sample nhẹ để vẽ nhanh)
# ═══════════════════════════════════════════════════════════════════════════════
vis_pts = pts_raw
if len(pts_raw) > 30000:
    vis_idx = np.random.choice(len(pts_raw), 30000, replace=False)
    vis_pts = pts_raw[vis_idx]

fig_raw = go.Figure(data=[
    go.Scatter3d(
        x=vis_pts[:, 0], y=vis_pts[:, 1], z=vis_pts[:, 2],
        mode='markers',
        marker=dict(size=1.0, color='lightblue', opacity=0.6)
    )
])
fig_raw.update_layout(
    title=f"{dataset_name} - Raw Point Cloud ({len(pts_raw):,} điểm)",
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
               aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig_raw.show()

# ═══════════════════════════════════════════════════════════════════════════════
# PREPROCESS: SUBSAMPLE + NORMALIZE
# ═══════════════════════════════════════════════════════════════════════════════
pts = preprocess_points(pts_raw, max_points=MAX_POINTS)

# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 3: Point Cloud sau preprocess (đây là input thực của CVR)
# ═══════════════════════════════════════════════════════════════════════════════
fig_proc = go.Figure(data=[
    go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=1.5, color='cyan', opacity=0.8)
    )
])
fig_proc.update_layout(
    title=f"{dataset_name} - Preprocessed ({len(pts):,} điểm, normalized)",
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
               aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig_proc.show()



# CVR cho Surface Modeling

In [ ]:
import subprocess
from sklearn.metrics import pairwise_distances
import numpy as np

CVR_BIN = "/content/experiments/cvr_train"

def compute_kernel_matrix(X1, X2, gamma):
    # Laplacian Kernel: k(x,y) = exp(-gamma * ||x-y||) (đúng theo paper 4.2)
    dists = pairwise_distances(X1, X2, metric='euclidean')
    return np.exp(-gamma * dists)

def save_libsvm_format(X, y, filepath):
    with open(filepath, 'w') as f:
        for i in range(len(X)):
            features = " ".join([f"{j+1}:{X[i,j]:.6f}" for j in range(X.shape[1])])
            f.write(f"{y[i]:.6f} {features}\n")

def read_cvr_model(model_path, X_train):
    with open(model_path, 'r') as f:
        lines = f.readlines()

    b = float(lines[0].strip())
    num_sv = int(lines[2].strip())

    SVs, coefs = [], []
    for line in lines[3:3+num_sv]:
        parts = line.strip().split()
        idx = int(parts[0])
        coef = float(parts[1])
        SVs.append(X_train[idx])
        coefs.append(coef)

    return np.array(SVs), np.array(coefs).reshape(-1, 1), b

def run_multiscale_cvr(X, sigmas=[0.2, 0.1, 0.05], C=1.0, mu=1e-4, epsilon=1e-4, data_dir="/content/experiments/data"):
    """Multi-scale CVR cho implicit surface modeling.

    Dữ liệu cần được normalize về [-0.5, 0.5] trước khi gọi.
    Sigma mặc định [0.2, 0.1, 0.05] phù hợp với tọa độ đã chuẩn hóa:
      - 0.2 = 20% kích thước object → crude shape
      - 0.1 = 10% → medium detail
      - 0.05 = 5% → fine detail
    """
    y_target = np.ones((len(X), 1))
    current_residual = y_target.copy()
    models = []
    total_svs = 0

    for i, sigma in enumerate(sigmas):
        print(f"--- Stage {i+1} / {len(sigmas)} | Sigma = {sigma} ---")
        gamma = 1.0 / sigma

        train_path = os.path.join(data_dir, f"temp_stage_{i}.libsvm")
        save_libsvm_format(X, current_residual.flatten(), train_path)

        model_path = os.path.join(data_dir, f"model_stage_{i}.txt")
        cmd = f"{CVR_BIN} {train_path} {X.shape[1]} {C} {mu} {gamma} {epsilon} {model_path}"
        subprocess.run(cmd, shell=True, check=True)

        SVs, coefs, b = read_cvr_model(model_path, X)
        n_sv = len(SVs)
        total_svs += n_sv
        print(f"-> Found {n_sv} Support Vectors.")

        # Đánh giá bằng batch để tránh bùng nổ RAM
        f_k = np.zeros((len(X), 1))
        batch_size = 2000
        for b_idx in range(0, len(X), batch_size):
            K = compute_kernel_matrix(X[b_idx:b_idx+batch_size], SVs, gamma)
            f_k[b_idx:b_idx+batch_size] = K @ coefs + b

        current_residual -= f_k
        models.append({'gamma': gamma, 'SVs': SVs, 'coefs': coefs, 'b': b})

    print(f"\nTổng: {total_svs} Support Vectors qua {len(sigmas)} stages.")

    def evaluate_implicit_surface(X_eval):
        f_total = np.zeros((len(X_eval), 1))
        batch_size = 5000  # Chia nhỏ RAM lúc vẽ lưới
        for i in range(0, len(X_eval), batch_size):
            X_batch = X_eval[i:i+batch_size]
            f_batch = np.zeros((len(X_batch), 1))
            for m in models:
                K = compute_kernel_matrix(X_batch, m['SVs'], m['gamma'])
                f_batch += K @ m['coefs'] + m['b']
            f_total[i:i+batch_size] = f_batch
        return f_total.flatten()

    return evaluate_implicit_surface

# ═══════════════════════════════════════════════════════════════════════════════
# CHẠY CVR (thông số đã hiệu chỉnh cho dữ liệu chuẩn hóa [-0.5, 0.5])
# ═══════════════════════════════════════════════════════════════════════════════
print(f"Huấn luyện Multi-Scale CVR trên {dataset_name}...")
f_implicit = run_multiscale_cvr(pts, sigmas=[0.1, 0.05, 0.02], C=10.0, mu=1e-4, epsilon=1e-5)

Huấn luyện Multi-Scale CVR trên Stanford Dragon...
--- Stage 1 / 3 | Sigma = 0.1 ---
-> Found 5700 Support Vectors.
--- Stage 2 / 3 | Sigma = 0.05 ---
-> Found 5107 Support Vectors.
--- Stage 3 / 3 | Sigma = 0.02 ---
-> Found 4895 Support Vectors.

Tổng: 15702 Support Vectors qua 3 stages.


In [ ]:
from skimage.measure import marching_cubes

def generate_mesh(implicit_func, pts, grid_res=100, level=1.0):
    bbox_min = pts.min(axis=0) - 0.05
    bbox_max = pts.max(axis=0) + 0.05

    x = np.linspace(bbox_min[0], bbox_max[0], grid_res)
    y = np.linspace(bbox_min[1], bbox_max[1], grid_res)
    z = np.linspace(bbox_min[2], bbox_max[2], grid_res)

    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    grid_points = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T

    print(f"\nĐánh giá Grid ({len(grid_points):,} điểm)...")
    F = implicit_func(grid_points)
    F_volume = F.reshape((grid_res, grid_res, grid_res))

    print("Chạy Marching Cubes...")
    vertices, faces, normals, _ = marching_cubes(F_volume, level=level,
                                                 spacing=(x[1]-x[0], y[1]-y[0], z[1]-z[0]))
    vertices += np.array([x[0], y[0], z[0]])
    return vertices, faces

vertices, faces = generate_mesh(f_implicit, pts, grid_res=100)
print(f"Hoàn thành! {len(vertices):,} đỉnh, {len(faces):,} mặt lưới.")



Đánh giá Grid (1,000,000 điểm)...
Chạy Marching Cubes...
Hoàn thành! 42,570 đỉnh, 85,212 mặt lưới.


In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color='lightblue', opacity=0.8,
        lighting=dict(ambient=0.4, diffuse=0.8, roughness=0.1, specular=0.5, fresnel=0.1),
        lightposition=dict(x=100, y=100, z=100)
    )
])

fig.update_layout(
    title=f"{dataset_name} - Implicit Surface (CVR)",
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()
